> **구분: 수업 후 복습·자습**

# 04. Transformer와 전체 비교 — 30분 + 정리 15분

이 자료는 AI를 처음 배우는 비전공 학부생을 기준으로 작성했습니다. 수식과 용어가 한 번에 이해되지 않아도 괜찮습니다. 먼저 **입력 → 계산 → 출력 → 결과 해석** 의 흐름을 따라 실행하고, 수업 뒤 상세 설명을 다시 읽으세요.

본인 실습 컨테이너의 Python 커널에서 환경 셀부터 실행합니다. 설명 → 예제 → 파일 TODO → 학습자 검증 → 참고 정답 순서입니다.

실행 순서와 저장 위치는 [04.LLM README](README.md), 낯선 용어는 [04.LLM 용어 찾아보기](GLOSSARY.md)에서 바로 확인할 수 있습니다.

- **TODO는 JupyterLab 파일 편집기에서 `exercises/*.py`를 수정하고 저장합니다.** 노트북이 소스를 덮어쓰지 않습니다.
- 미완성 검증이 `AssertionError`로 실패하는 것은 정상입니다. 실패 메시지를 읽고 TODO를 고친 뒤 검증 셀을 다시 실행하세요.
- `load_op`는 저장된 파일을 다시 읽으므로 보통 커널을 재시작할 필요가 없습니다. 단, 일반 `import`로 읽은 모듈은 재시작 또는 reload가 필요할 수 있습니다.
- 완료한 앞 교시 TODO를 다음 훈련에서도 사용합니다. 참고 정답은 별도 `solutions/` 경로에 있으며, 강사가 안내하는 시점에 비교하고 자신의 코드로 재검증합니다.
- 표기 `[B,T,H]`는 텐서의 **shape** 입니다. `B`는 한 번에 처리하는 샘플 수(batch), `T`는 순서 길이(time/sequence), `H`는 내부 표현 크기(hidden), `V`는 token 종류 수(vocabulary)입니다.
- 컨테이너를 재생성하면 패키지가 포함된 최신 이미지가 필요하며, 소스와 `Data`는 호스트에 보존됩니다.

오류가 나면 가장 먼저 현재 노트북의 앞 셀을 순서대로 실행했는지, TODO 파일을 저장했는지, 오류가 난 텐서의 `shape`, `dtype`, `device`가 무엇인지 확인하세요.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 현재 실습의 입력·저장·실행 범위

열 파일은 `/workspace/04.LLM/04_transformer.ipynb`입니다. 첫 환경 셀은 `ROOT`, `COURSE`, `DATA`, `OUTPUTS`, `check`, `load_op`, `run_training`을 **이 커널에 다시 정의** 합니다. Python 경로와 PyTorch·CUDA 출력에서 CUDA가 `True`인지 확인하고 같은 Jetson의 다른 사용자와 GPU 실행 순서를 맞춥니다. 이 숫자 모델 훈련에는 Ollama 서버가 필요하지 않습니다.

입력은 `/workspace/Data/datasets/04.LLM/digits.json`과 `.sha256`입니다. 이 데이터 640개를 train 512개·validation 128개로 나누며, PAD/BOS/EOS=0/1/2, 숫자 0~9의 token ID=3~12입니다. 파일이 없으면 배포 데이터가 포함된 개인 자료 폴더인지 확인합니다. 이전 훈련 checkpoint 없이 새 훈련을 시작합니다.

`exercises/*.py`는 Jupyter 파일 편집기에서 수정·저장하세요. 노트북의 파일 출력 셀은 읽기 전용이고 `check(..., 'exercise')`는 저장한 최신 파일을 읽습니다. 결과는 `/workspace/Data/outputs/04.LLM/<실행명>/metrics.json`·`loss.csv`, 자신의 가중치는 `/workspace/Data/checkpoints/users/04.LLM/`에 남습니다. `run_training`은 새 결과 폴더를 만들고 반환한 `report['checkpoint']`는 `Data` 기준 상대 경로입니다. 다른 노트북의 `report` 변수는 이어지지 않습니다.

<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

attention만 사용하면 입력 순서를 별도로 알려 주어야 합니다. Transformer는 위치 정보와 attention을 결합하며 생성 시 미래 정답을 보지 못하도록 mask를 둡니다. 이 자습의 작은 encoder-decoder와 배포 Gemma의 구조를 구분합니다.

### 전체 과정에서 내 코드가 하는 일

```text
token·위치 정보
  ↓
encoder/decoder attention
  ↓
미래 위치 차단
  ↓
next-token loss
  ↓
자유 생성과 검증
```

causal mask와 tensor 모양을 구현하고 PAD 처리와 미래 위치 차단을 구분합니다. 사용하는 API에서 True가 차단을 뜻하는지 현재 코드 계약을 따라 확인합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 미래 정답을 볼 수 있게 두면 loss가 낮아져도 좋은 모델인가요?

**설명:** 아닙니다. 생성 시 없는 정보를 학습·평가에 노출한 것입니다. 현재 위치는 허용하고 미래 위치는 막는 경계를 확인합니다. padding mask는 의미 없는 패딩 위치를 다루므로 목적이 다릅니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| mask shape/dtype 오류·비교 결과 누락 | causal mask의 미래 위치 차단 방향을 검사하고 loss/shift/mask TODO를 확인합니다. 비교할 모델별 같은 profile의 완료 metrics가 있는지 확인합니다. | 해당 check(..., exercise) 통과 |
| `ModuleNotFoundError: common` / 다른 함수가 로딩됨 | 작업 경로와 커널을 확인합니다. 다른 과목 common이 남으면 저장 후 해당 커널을 재시작하고 초기 셀부터 실행합니다. | 현재 04.LLM 모듈 import 성공 |
| 훈련 timeout·OOM·loss NaN | 직전 stdout/stderr와 설정을 보존합니다. 작은 quick profile부터 확인하고 중복 실행·입력 유한성·device를 점검합니다. | 작은 훈련 완료, 유한 loss와 metrics 저장 |

**예시에 없는 오류를 확인할 곳:** [LLM README](README.md), [contracts 검사](common/contracts.py), exercises/solutions의 해당 연산. 실행이 출력한 output 경로의 metrics.json과 터미널 오류를 확인하세요. [PyTorch 이슈](https://github.com/pytorch/pytorch/issues)는 버전과 최소 tensor 예제로 검색합니다.


[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 후 복습·자습**

본수업 훈련 120분은 [00_training_core.ipynb](00_training_core.ipynb)에서 진행합니다. 이 노트북 전체는 강사 선택 시연·자습이며 여기에 적힌 시간은 별도 자습 예상 시간입니다. 다섯 모델 구현 완료는 본수업 통과 조건이 아닙니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 작은 모델의 훈련 전후 예측 또는 Ollama 응답을 저장합니다.
- **단원 마지막 연결 질문:** 직접 훈련한 숫자 과제와 이미 학습된 모델 사용을 구분합니다. 제공한 작업 지침에 없는 질문에는 근거가 없다고 말하는지 확인합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


> **구분: 수업 후 복습·자습**

## 시간·목표

핵심 지도 5분 → causal mask TODO 7분 → 훈련 8분 → 결과 해석 10분 → 최신 5개 실행 비교·정리 15분. 선행: loss·shift.

### 자습 진행 경로

1. 환경과 바로 아래 **45분 수업 핵심 지도** 를 읽습니다.
2. 4절의 causal mask 표를 확인하고 mask TODO·검증을 완료합니다.
3. Transformer quick 훈련과 5절 결과를 확인합니다.
4. 6절 비교 셀에서 모델별 최신 exercise/quick 실행 한 개씩만 확인합니다.
5. 1–3절 상세 구조, 7절 사전학습, 8절 오류 표는 수업 후 복습합니다.

각 셀의 **구분** 표시를 확인하세요. `수업 후 복습·자습` 셀은 건너뛰어도 다음 수업 핵심 셀을 실행할 수 있습니다.

### 앞선 loss TODO의 위치와 현재 필요한 내용

원래 설명은 [00_training_core.ipynb의 「수업 핵심 TODO — loss」](00_training_core.ipynb)와 [01_training_basics.ipynb의 「자습 구현 TODO — loss」](01_training_basics.ipynb)에 있습니다. 두 노트북이 편집하는 실제 파일은 **[exercises/loss.py](exercises/loss.py)** 하나입니다. 이전 커널을 열 필요 없이 이 파일의 `token_loss(logits, targets)`를 아래 계약으로 완성·저장하세요.

- 입력 logits는 raw 실수 점수 `[B,T,V]`, targets는 같은 장치의 정수 class 번호 `[B,T]`(값 `0..V-1`)입니다.
- 마지막 후보 축 V를 남겨 logits를 `[B*T,V]`로, targets를 같은 순서로 `[B*T]`로 펼칩니다. `reshape(-1, logits.shape[-1])`와 `reshape(-1)`의 역할을 구분합니다.
- `F.cross_entropy`에 두 값을 전달하고 평균 loss인 0차원 텐서를 반환합니다. softmax를 미리 적용하거나 `.item()`, `.detach()`로 gradient 연결을 끊지 않습니다.
- 모든 점수가 같은 후보 4개라면 loss는 약 `1.3863`이지만 이 값을 고정 반환하면 안 됩니다. 실제 검사에서는 정답 점수를 낮출 때 loss 증가와 backward의 비영 기울기까지 확인합니다.
- 이 노트북의 아래 `check('loss', 'exercise')`가 저장한 파일을 다시 읽습니다. 통과 후 훈련하세요. 참고 정답은 [solutions/loss.py](solutions/loss.py)에 있으며 이를 실행하는 것만으로 exercise 파일은 완성되지 않습니다.

### 앞선 shift TODO의 위치와 현재 필요한 내용

원래 설명은 [03_seq2seq_attention.ipynb의 「자습 구현 TODO — shift」](03_seq2seq_attention.ipynb), 실제 수정 파일은 **[exercises/shift.py](exercises/shift.py)** 입니다. `shift_right(targets, bos=1)`은 길이 T≥1의 정수 텐서 `[B,T]`를 받아 같은 shape·dtype·device의 **새 텐서** 를 반환합니다. `torch.empty_like`로 새 공간을 만들고 첫 열에는 bos, 두 번째 열부터는 `targets[:, :-1]`을 복사합니다. 입력 자체는 바꾸지 않습니다. `[[4,5,6]]`은 `[[1,4,5]]`, 길이 1이고 bos=7이면 `[[7]]`이어야 합니다.

이 처리는 현재 정답을 보지 않고 직전 정답을 입력하는 teacher forcing 준비입니다. Transformer 훈련은 loss·shift·현재 mask를 사용하며 **attention.py 구현은 Transformer 자체 훈련의 선행 조건이 아닙니다.** 아래 훈련 셀의 세 검사를 통과한 뒤 시작합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
from pathlib import Path
import importlib, json, subprocess, sys, time
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '04.LLM/common').is_dir())
COURSE = ROOT / '04.LLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import DATA, OUTPUTS
from common.contracts import check
from common.loading import load_op
import torch
OUTPUTS.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable, 'PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '훈련은 Jetson GPU에서 수행합니다. 강사에게 환경 점검을 요청하세요.'
def run_training(kind, variant='exercise'):
    before = {p.resolve() for p in OUTPUTS.glob('*/metrics.json')}
    command = [sys.executable, str(COURSE / 'scripts/train.py'), '--model', kind, '--variant', variant]
    result = subprocess.run(command, text=True, capture_output=True, timeout=300)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    result.check_returncode()
    created = [p for p in OUTPUTS.glob('*/metrics.json') if p.resolve() not in before]
    assert len(created) == 1, f'새 metrics.json이 정확히 1개여야 합니다: {created}'
    metrics_path = created[0]
    report = json.loads(metrics_path.read_text())
    assert report['kind'] == kind and report['variant'] == variant and report['profile'] == 'quick'
    report['_metrics_path'] = str(metrics_path.relative_to(DATA))
    print('검증 loss:', report['before']['loss'], '→', report['after']['loss'])
    if kind in ('rnn','lstm'):
        print('token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    else:
        print('자유 생성 token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('자유 생성 시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    print('실행 경로:', report['run_path'])
    return report


> **구분: 수업 후 복습·자습**

## 45분 수업 핵심 지도

- Transformer는 attention, FFN, residual connection과 normalization을 층으로 결합합니다.
- 입력 순서를 구분하기 위해 token embedding에 position embedding을 더합니다.
- Decoder의 causal mask는 현재 위치가 미래 정답을 보지 못하게 합니다. bool mask는 이 API에서 `True=차단`입니다.
- 이번 모델은 encoder-decoder Transformer이며, Gemma 같은 decoder-only LLM과 전체 구조가 같지는 않습니다.
- Transformer quick 훈련 뒤에는 각 kind의 최신 `variant=exercise`, `profile=quick` 결과만 골라 비교합니다.


> **구분: 수업 후 복습·자습**

## 1. Transformer가 바꾼 시퀀스 처리 방식

RNN 계열은 hidden state를 넘기며 위치를 순서대로 처리합니다. **Transformer** 는 attention으로 위치 사이의 관계를 직접 계산합니다. 훈련에서는 여러 위치의 계산을 병렬화하기 쉬워 대규모 데이터와 하드웨어를 활용하는 데 유리합니다. 다만 attention의 기본 계산량과 메모리는 시퀀스 길이의 제곱에 비례하므로 긴 문맥이 공짜로 처리되는 것은 아닙니다.

Transformer의 한 블록에는 보통 다음 요소가 있습니다.

- **Multi-head attention** : Q/K/V를 여러 작은 공간(head)으로 나누어 서로 다른 관계를 동시에 학습합니다. 각 head가 반드시 사람이 붙인 하나의 문법 역할과 일치하는 것은 아닙니다.
- **Feed-forward network(FFN)** : 각 token 위치에 같은 작은 완전연결 신경망을 적용해 표현을 변환합니다.
- **Residual connection** : 블록 입력을 변환 결과에 더해 정보와 gradient가 흐를 경로를 만듭니다.
- **Layer normalization** : 각 token 표현의 수치 규모를 정규화해 안정적인 훈련을 돕습니다.

Attention만 하나 넣은 모델을 곧바로 완전한 Transformer라고 부르지는 않습니다. 이 요소들이 층을 이루어 함께 작동합니다.


> **구분: 수업 후 복습·자습**

## 2. Self-attention, cross-attention, encoder와 decoder

<!-- procedure:lesson-14-9e7b5f:start -->
![Transformer 입력에서 다음 token까지](../교재그림/절차/lesson-14.svg)

미래 token을 가리는 위치와 다음 token을 만드는 위치를 구분합니다. mask TODO의 입출력 형상을 확인하세요.
<!-- procedure:lesson-14-9e7b5f:end -->


- **self-attention** : 같은 시퀀스에서 Q, K, V를 만들어 그 시퀀스 안의 위치끼리 정보를 교환합니다.
- **cross-attention** : decoder의 표현을 Q로, encoder 출력을 K/V로 사용해 출력이 입력의 관련 위치를 참조합니다.

이 실습은 숫자 역순 변환을 위해 **encoder-decoder Transformer** 를 사용합니다. encoder는 source 전체를 self-attention으로 읽습니다. decoder는 지금까지의 target 문맥에 masked self-attention을 적용한 뒤 encoder memory를 cross-attention으로 참조합니다.

Gemma와 많은 현대 텍스트 LLM은 주로 **decoder-only Transformer** 입니다. 하나의 token 시퀀스에 causal self-attention을 사용해 다음 token을 계속 예측합니다. 따라서 이번 작은 모델은 Transformer의 핵심 연산을 보여 주지만 Gemma의 전체 구조와 동일하지 않습니다.


> **구분: 수업 후 복습·자습**

## 3. 위치 정보가 필요한 이유

Attention 계산만 보면 입력 token의 순서를 자동으로 알 수 없습니다. 같은 token 벡터 집합의 순서를 바꿔도 별도 위치 신호가 없다면 순서를 구분하기 어렵습니다. 그래서 token embedding에 **position encoding/embedding** 을 결합합니다.

이 실습은 위치 0, 1, 2, ...마다 학습 가능한 벡터를 찾아 token embedding에 더합니다.

```text
token embedding    [B,T,H]
position embedding [1,T,H]  ← batch 전체에 broadcast
합                 [B,T,H]
```

실제 모델에는 고정 sinusoidal encoding, 학습형 absolute position, RoPE 같은 여러 방식이 있습니다. 위치 정보를 넣는 방법이 달라도 'token 내용과 순서를 함께 표현한다'는 목적은 같습니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
from common.models import TinyTransformer
tiny = TinyTransformer(load_op('mask','solution').causal_mask)
print(tiny)
demo_tokens = torch.tensor([[3,4,5,6,7],[7,6,5,4,3]])
print('token:', demo_tokens.shape)
print('token+position embedding:', tiny.embed(demo_tokens).shape)

> **구분: 수업 후 복습·자습**

## 4. causal mask: 미래 정답을 가리는 장치

행은 현재 query 위치, 열은 참조할 key 위치입니다. **열 > 행** 인 미래 위치만 차단합니다.
이 실습의 `nn.Transformer` bool mask에서는 True가 차단입니다. 다른 attention API의 bool 의미와 혼동하지 마세요.
행 0은 열 0만, 행 1은 열 0·1만 볼 수 있습니다. encoder 입력에는 이 causal mask를 적용하지 않습니다.

길이 4의 올바른 bool mask는 다음과 같습니다. `T`가 차단(True), `.`가 허용(False)입니다.

```text
key 열 →     0 1 2 3
query 행 0   . T T T
          1   . . T T
          2   . . . T
          3   . . . .
```

대각선은 현재 위치이므로 허용하고, 대각선 위쪽만 차단합니다. Teacher forcing으로 정답 시퀀스 전체를 한 번에 GPU에서 계산해도 이 mask 덕분에 각 위치는 미래 정답을 참조할 수 없습니다.

mask의 의미는 API마다 다릅니다. 이 자료의 `torch.nn.Transformer`에 전달하는 bool `tgt_mask`는 **True가 차단** 입니다. 일부 scaled-dot-product attention API에서는 bool의 의미가 반대일 수 있고, float mask는 차단 위치에 `-inf`를 더하기도 합니다. 기억에 의존하지 말고 사용하는 함수의 문서와 작은 예제로 검증하세요.


> **구분: 수업 후 복습·자습**

## 자습 구현 TODO — mask

수정 파일: **[04.LLM/exercises/mask.py](exercises/mask.py)**

`[length,length]` bool mask를 생성합니다. 미래 위치만 True이고 대각선은 False입니다.
힌트: `torch.ones(..., dtype=torch.bool, device=device)`와 `torch.triu(..., diagonal=1)`.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `causal_mask(length, device)`는 양의 길이와 장치를 받아 `[length,length]` bool 텐서를 반환합니다. 행은 현재 query 위치, 열은 참조할 위치이며 True는 가려야 하는 미래입니다.
**작성 흐름 힌트:** bool 정사각 배열 → 대각선 바로 위부터 상삼각형만 True. length=3이면 `[[False,True,True],[False,False,True],[False,False,False]]`입니다.
**설명할 지점:** “대각선까지 가리면 현재 위치도 못 봅니다. True가 허용이 아니라 차단이라는 이 API의 약속을 확인하세요.”
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'exercises/mask.py').read_text())

In [ ]:
# 구분: 수업 후 복습·자습 셀
check('mask', 'exercise')
print('learner_verification=PASS')

> **구분: 수업 후 복습·자습**

### 참고 정답

자습에서는 필요할 때 정답과 해설을 열어 비교하고 자신의 코드로 재검증합니다. 아래 실행은 자신의 TODO를 변경하지 않습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'solutions/mask.py').read_text())

### 정답 해설 — 입력부터 결과까지

**미래만 가리기:** `triu(...,diagonal=1)`은 대각선 바로 위부터 True로 만듭니다. 이 수업 mask에서 True는 참조를 막는 위치입니다. 길이 3이면 첫 행은 `[False,True,True]`, 둘째는 `[False,False,True]`, 셋째는 모두 False입니다.

대각선 자체는 현재 위치이므로 허용합니다. `diagonal=0`으로 쓰면 현재 위치도 가립니다. device를 입력 연산과 맞춰 CPU/GPU 혼합 오류를 피합니다.

**확인 질문:** 0번째 출력이 2번째 정답을 볼 수 있다면 다음 token 예측 평가가 왜 공정하지 않나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/mask.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
import torch

def causal_mask(length, device):
    # 행 i에서 열 j>i인 미래 위치만 True(차단); 자기 위치는 허용합니다.
    return torch.triu(torch.ones(length,length,dtype=torch.bool,device=device), diagonal=1)
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
check('mask', 'solution')
print('solution_verification=PASS')

> **구분: 수업 후 복습·자습**

### causal mask와 padding mask는 목적이 다릅니다

- **causal mask** `[T,T]`: 한 출력 위치가 미래 출력 위치를 보지 못하게 합니다.
- **padding mask** `[B,T]`: 길이를 맞추려고 넣은 PAD 위치를 attention에서 무시합니다.

이번 데이터는 입력 길이가 모두 같아 padding mask를 생략합니다. 실제 자연어 batch에서는 두 mask가 동시에 필요할 수 있습니다. 둘 다 무언가를 가리지만 shape와 목적이 다릅니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
mask_demo = load_op('mask','solution').causal_mask(6, torch.device('cpu'))
print(mask_demo)
print('shape:', mask_demo.shape, 'dtype:', mask_demo.dtype)

In [ ]:
# 구분: 수업 후 복습·자습 셀
for required_op in ('loss', 'shift', 'mask'):
    check(required_op, 'exercise')
transformer_report = run_training('transformer')

> **구분: 수업 후 복습·자습**

## 5. 결과와 체크포인트

훈련 결과에서 loss 감소와 자유 생성 정확도 개선을 함께 확인합니다. Transformer도 teacher forcing으로 loss를 계산하므로 실제 생성 지표를 따로 봐야 합니다. 짧은 규칙에서 100%가 나와도 일반적인 언어 능력이 생긴 것은 아닙니다.

**checkpoint** 는 특정 시점의 모델 상태를 파일로 저장한 것입니다. `state_dict`에는 층별 weight와 bias 텐서가 들어 있습니다. 다시 불러올 때 같은 코드와 같은 shape의 모델 구조가 필요합니다. 훈련 스크립트는 새 모델에 가중치를 로드한 뒤 동일 입력의 예측이 같은지 검사합니다.

이번 체크포인트는 **추론 재현용** 입니다. optimizer state, 현재 step의 세부 상태와 난수 생성기 상태를 모두 저장하지 않으므로 중단 지점부터 정확히 이어 훈련하는 용도가 아닙니다. checkpoint는 신뢰하는 출처의 파일만 로드해야 하며, 이 자료는 가능한 범위에서 `weights_only=True`를 사용합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
print(transformer_report['checkpoint'])
print(transformer_report['after']['examples'])

> **구분: 수업 후 복습·자습**

## 6. 15분 전체 정리

아래 표에서 각 실행의 검증 지표와 학습 시간을 비교합니다. RNN/LSTM은 지연 복사, 나머지는 역순 변환이므로 **서로 다른 과제의 정확도를 순위로 비교하지 않습니다.**
짧은 훈련에서 보이는 개선과 모델 간 차이를 설명하고, loss만으로 판단하지 마세요.

| 구조 | 시퀀스 정보 전달 | 이번 실습에서 볼 점 |
|---|---|---|
| RNN | hidden state를 시점마다 전달 | 반복 상태와 짧은 기억 |
| LSTM | hidden + gated cell state | gate가 있는 기억 통로 |
| Seq2Seq(GRU) | encoder 마지막 state로 decoder 시작 | 입력/출력 시퀀스 분리, teacher forcing |
| Attention Seq2Seq | decoder가 encoder의 모든 위치를 가중합 | 고정 context 병목 완화 |
| Transformer | self/cross-attention과 FFN을 층으로 구성 | 위치 표현, causal mask, 병렬 훈련 |

파라미터 수, step 수와 과제가 모두 같지 않으므로 아래 출력은 엄밀한 모델 벤치마크가 아닙니다. 같은 과제인 Seq2Seq·Attention·Transformer끼리도 단일 seed의 작은 데이터 결과입니다.

비교 셀은 `Data/outputs/04.LLM/<run>/metrics.json`을 읽되, 각 kind에서 `variant=exercise`, `profile=quick`, quick 설정의 step/seed와 정확히 일치하는 결과만 필터링합니다. 그중 파일 수정 시각이 가장 최신인 **한 개** 만 선택합니다. 과거 `solution`, `selfstudy`와 같은 모델의 중복 실행은 출력하지 않습니다. 표시되는 `run`과 `path`로 어느 결과를 읽었는지 확인하세요.

### 이전 결과가 없는 경우의 위치와 재생성 방법

이전 결과는 `/workspace/Data/outputs/04.LLM/<실행명>/metrics.json`에 있습니다. [02_rnn_lstm.ipynb의 「훈련 전 확인」](02_rnn_lstm.ipynb)은 loss 검증 후 `run_training('rnn')`, `run_training('lstm')`를 실행합니다. [03_seq2seq_attention.ipynb의 shift·attention 검증 뒤 훈련 셀](03_seq2seq_attention.ipynb)은 loss·shift·attention 검증 후 `run_training('seq2seq')`, `run_training('attention')`를 실행합니다. **현재 노트북에도 같은 `run_training` 도우미가 정의되어 있으므로** 전체 비교를 자습으로 완성하려면 현재 커널에서 그 네 호출을 실행할 수 있습니다. Attention 모델만 `exercises/attention.py`가 추가로 필요합니다. 계약은 query `[B,1,H]`, keys/values `[B,T,H]`에서 `QKᵀ/sqrt(H)`→마지막 축 softmax→value 가중합을 계산해 `(context [B,1,H], weights [B,1,T])`를 반환하는 것입니다.

이 노트북만 시작한 경우 아래 셀은 없는 모델을 **미측정** 으로 표시하고 현재 Transformer 결과를 계속 보여 줍니다. 측정하지 않은 값을 참고값이나 0으로 채우지 않습니다. 5개 전체 비교를 완료하려는 자습에서는 빠진 모델을 위 순서로 훈련한 뒤 비교 셀을 다시 실행하세요.


In [ ]:
# 구분: 수업 후 복습·자습 셀
quick = json.loads((COURSE / 'configs/quick.json').read_text())
kinds = ('rnn','lstm','seq2seq','attention','transformer')
selected = {}
for kind in kinds:
    candidates = []
    for metrics_path in OUTPUTS.glob('*/metrics.json'):
        report = json.loads(metrics_path.read_text())
        if not (report.get('kind') == kind
                and report.get('variant') == 'exercise'
                and report.get('profile') == 'quick'
                and report.get('steps') == quick[kind]
                and report.get('seed') == quick['seed']):
            continue
        expected = DATA / report['run_path'] / 'metrics.json'
        if expected.resolve() != metrics_path.resolve():
            raise AssertionError(f"보고서 경로 불일치: {metrics_path} != {expected}")
        candidates.append((metrics_path.stat().st_mtime_ns, metrics_path, report))
    if not candidates:
        print(f'{kind}: 미측정 — 현재 노트북의 「이전 결과가 없는 경우의 위치와 재생성 방법」을 확인하세요.')
        continue
    _, metrics_path, report = max(candidates, key=lambda item:item[0])
    selected[kind] = report
    accuracy_label = 'token 정확도' if kind in ('rnn','lstm') else '자유 생성 token 정확도'
    sequence_label = '시퀀스 전체 일치율' if kind in ('rnn','lstm') else '자유 생성 시퀀스 전체 일치율'
    print(f"{kind}: variant={report['variant']} profile={report['profile']} run={report['run_name']}")
    print('  path=', report['run_path'], 'metrics=', metrics_path.relative_to(DATA))
    print('  steps=', report['steps'], 'parameters=', report['parameters'],
          'seconds=', round(report['training_seconds'],2),
          accuracy_label+'=', round(report['after']['token_accuracy'],3),
          sequence_label+'=', round(report['after']['sequence_accuracy'],3))
print('전체 비교 완료:', set(selected) == set(kinds))


> **구분: 수업 후 복습·자습**

## 7. 작은 모델 훈련에서 사전학습 LLM 사용으로

지금까지는 무작위 파라미터에서 시작해 작은 과제 데이터로 모델을 훈련했습니다. 실제 LLM 서비스는 보통 여러 단계를 거친 모델을 사용합니다.

1. **사전학습(pretraining)** : 매우 많은 텍스트에서 다음 token 예측 같은 목표로 일반적인 언어 패턴과 지식을 학습합니다. 데이터·계산 비용이 큽니다.
2. **미세조정(fine-tuning)** : 특정 분야 데이터나 원하는 작업 형식에 맞춰 사전학습 파라미터를 추가로 조정합니다.
3. **instruction tuning** : 지시와 적절한 답 형식의 예제로 사용자의 요청을 따르는 능력을 조정하는 미세조정 방식입니다.
4. **선호 정렬(alignment)** : 사람 또는 모델의 선호 신호를 이용해 답변 행동을 더 조정할 수 있습니다. 구체적인 알고리즘은 모델마다 다릅니다.
5. **추론(inference)** : 완성된 파라미터를 고정하고 프롬프트 뒤의 token을 순차적으로 생성합니다.

5–6번의 Ollama 실습은 1–4단계를 Jetson에서 수행하지 않습니다. 미리 준비한 Gemma 가중치를 읽어 **추론** 합니다. 모델 파일을 다운로드하는 것, 메모리에 적재하는 것, 미세조정하는 것, 대화하는 것은 서로 다른 작업입니다.


> **구분: 수업 후 복습·자습**

## 8. 자주 생기는 오해와 오류

- **"Transformer는 문장을 한 번에 생성한다"** : 훈련 시 여러 위치를 병렬 계산할 수 있지만, 일반적인 자기회귀 생성은 직전 출력에 의존해 token을 한 개씩 만듭니다.
- **"mask를 쓰면 해당 token이 입력에서 삭제된다"** : attention score에서 특정 참조 경로를 차단합니다. 텐서의 행이나 열 자체를 없애는 것은 아닙니다.
- **"position 값이 token 번호와 같은 의미다"** : token embedding과 position embedding은 별도 학습 테이블이고 더해서 같은 hidden 공간에 표현합니다.
- **mask가 반대로 보임** : 사용하는 API에서 bool True의 의미를 확인하고 길이 4 예제로 출력합니다.
- **checkpoint load 시 missing/unexpected key** : 저장한 모델과 생성한 모델의 층 이름·구조가 다릅니다.
- **Ollama로 넘어간 뒤 CUDA 메모리가 부족함** : 훈련은 별도 프로세스라 정상 종료하면 자원이 반환됩니다. 실행 중인 훈련 프로세스와 다른 사용자의 GPU 작업을 확인합니다.


> **구분: 수업 후 복습·자습**

## 완료·자습

- 현재 Transformer의 TODO 검증·훈련과 Data 결과·체크포인트 위치를 확인합니다.
- 전체 비교 자습은 5개 모델 결과를 모두 확인했을 때 완료이며, 없는 결과는 미측정으로 남깁니다.
- teacher forcing와 자유 생성, causal mask와 입력 누수의 관계를 설명합니다.
- 자습(30분): selfstudy profile로 훈련하고 seed·step·파라미터 수를 기록해 비교합니다.
- 훈련 프로세스가 종료된 뒤 Ollama 실습으로 넘어갑니다. 같은 Jetson의 다른 사용자와 GPU 사용 순서를 맞춥니다.

참고: [PyTorch Transformer](https://docs.pytorch.org/docs/stable/generated/torch.nn.Transformer.html)
